In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 10.0 MB/s eta 0:00:00


In [2]:
import anthropic
import time
import csv
from tqdm import tqdm
import argparse
from openai import OpenAI

## **CONFIGURATION**

In [3]:
OPENAI_API_KEY    = "OPENAI_API_KEY"
ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"

MODELS = {
    "gpt-5.4":           "openai",
    "claude-opus-4-6":   "anthropic"
}

# Reflection settings
MAX_REFLECT  = 2     # fixed number of reflection rounds
DELTA        = 0.10  # minimum score change to continue reflecting

openai_client    = OpenAI(
    api_key = OPENAI_API_KEY
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [4]:
def call_llm(prompt, model):

    provider = MODELS.get(model)

    if provider is None:
      print(f"  Unknown model: {model}")
      return None

    if provider == "openai":
      response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
      return response.choices[0].message.content

    elif provider == "anthropic":
      response = anthropic_client.messages.create(
            model=model,
            temperature=0,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
      )
      return response.content[0].text


    return None

##AUTO-COT STEP GENERATION

In [5]:
def generate_cot_steps_no_gold(model):
    """
    Generate CoT steps for ATTRIBUTE evaluation
    WITHOUT gold reference.
    Used by S2 and S3 (S3 reuses S2's steps).
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate the ATTRIBUTES in a PlantUML candidate diagram given only
a problem description.

The evaluator will apply this correctness rule:
- Supported by problem description correct → CORRECT
- Not supported by problem description → INCORRECT
- Real-world knowledge is NOT a valid source.

Generate steps that cover:
- How to identify expected attributes per class  from the problem description
- How to measure Attribute Completeness (0-1):
  fraction of expected attributes present in candidate
  (only within matched classes)
- How to measure Attribute Correctness (0-1):
  fraction of candidate attributes that are accurate,
  meaning correct name, correct data type
  (int, String, List<T>, Date, etc.),
- How to handle naming variants
  (e.g. firstName vs first_name vs fname)
- How to handle extra/hallucinated attributes
  (present in candidate but not expected)
- How to handle missing attributes
  (expected but absent in candidate)

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (no gold) using {model}...")
    steps = call_llm(prompt, model)

    with open("cot_steps_attributes_no_gold.txt", "w",
              encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_attributes_no_gold.txt")

    return steps


def generate_cot_steps_with_gold(model):
    """
    Generate CoT steps for ATTRIBUTE evaluation
    WITH gold reference.
    Used by S5 and S6.
    """
    prompt = """You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the ATTRIBUTES in a PlantUML candidate diagram against
a gold reference diagram.

The evaluator will apply this correctness rule:
- Matched AND stereotype correct → CORRECT
- Not matched but supported by problem → CORRECT
- Not matched AND not supported by problem → INCORRECT
- Real-world knowledge is NOT a valid source.

Generate steps that cover:
- How to match candidate classes to gold classes first
  (exact name match, then semantic match)
- For each matched class pair, how to match candidate
  attributes to gold attributes
- How to measure Attribute Completeness (0-1):
  matched attributes / total gold attributes
  (across all matched classes)
- How to measure Attribute Correctness (0-1):
  correct attributes / total candidate attributes
  where correct means: name matched AND data type
  matches the gold (int, String, List<T>, Date, etc.)
- How to handle naming variants
  (e.g. firstName vs first_name vs fname)
- How to handle extra/hallucinated attributes
  (in candidate but not in gold)
- How to handle missing attributes
  (in gold but not in candidate)
- How to handle attributes in unmatched classes
  (classes that exist in candidate but not in gold)

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (with gold) using {model}...")
    steps = call_llm(prompt, model)

    with open("cot_steps_attributes_with_gold.txt", "w",
              encoding="utf-8") as f:
        f.write(steps)
    print("  Saved to cot_steps_attributes_with_gold.txt")

    return steps



## Reflection

In [6]:
def needs_reflection(round_num, current_scores,previous_scores=None):
    """
    Decide whether to run a reflection round.

    Rules:
    - Round 1: always runs (previous_scores is None,
      no delta to compare yet)
    - Round 2: only runs if max score delta between
      initial scores and round 1 scores > DELTA
    - Never exceeds MAX_REFLECT rounds

    Args:
        round_num:       current round number (1-based)
        current_scores:  scores from most recent round
        previous_scores: scores from the round before
                         (None for round 1)
    """
    # Never exceed max reflection rounds
    if round_num > MAX_REFLECT:
        return False

    # Round 1 always runs — no previous scores to compare
    if previous_scores is None:
        return True

    # Round 2+ — only run if scores are still changing
    deltas = [
        abs(current_scores[k] - previous_scores[k])
        for k in current_scores
        if current_scores.get(k) is not None
        and previous_scores.get(k) is not None
    ]

    if not deltas:
        return False

    return max(deltas) > DELTA

## PROMPT

In [7]:

def s1_prompt(problem, candidate):
    """S1 — Simple baseline. No CoT, no gold."""
    return f"""You are evaluating the ATTRIBUTES in a PlantUML
candidate diagram.

Definitions:
- Completeness (0-1): fraction of expected attributes present in the candidate diagram.
- Correctness (0-1): fraction of candidate attributes that are accurate (correct name, correct data type).

Correctness Rule:
The ONLY valid reference is the problem description.
- Compare against the problem description.
- Do NOT use real-world domain knowledge as a source.


Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


def s2_prompt(problem, candidate, cot_steps):
    """
    S2 and S3 base scoring prompt.
    S3 uses this for initial scoring, then adds reflection.
    No gold reference.
    """
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


def s3_reflection_prompt(problem, candidate,
                                cot_steps, previous_scores,
                                round_num):
    """
    S3 — Reflection prompt. No gold reference.
    Round 1 always runs. Round 2 only if delta > DELTA.
    """
    comp = previous_scores['attr_completeness']
    corr = previous_scores['attr_correctness']

    return f"""You previously evaluated the ATTRIBUTES in this
diagram (reflection round {round_num}) and produced:

- Attribute Completeness: {comp}
- Attribute Correctness:  {corr}

Please reflect carefully using ONLY the problem
description as your reference:

For Attribute Completeness:
1. Re-read the problem description carefully.
2. For each expected class, list the attributes you
   believe it should have based on the problem.
3. Check which of those attributes are present
   in the candidate diagram.
4. Recompute: completeness = present / expected
5. Revise your score if needed.

For Attribute Correctness:
1. List every attribute in the candidate diagram.
2. For each one, verify:
   - Is the name appropriate for this problem?
   - Is the data type appropriate (int, String, etc.)?
3. Recompute: correctness = correct / total candidate
4. Revise your score if needed.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


def s4_prompt(problem, gold, candidate):
    """S4 — Reference only. No CoT."""
    return f"""You are evaluating the ATTRIBUTES in a PlantUML
candidate diagram against a gold reference.

Definitions:
- Completeness (0-1): fraction of gold attributes present in the candidate, across all matched classes.
  Formula: matched attributes / total gold attributes
- Correctness (0-1): fraction of candidate attributes that are accurate compared to gold.
  Formula: correct / total candidate attributes
  A candidate attribute is CORRECT if it is matched to a gold attribute AND has the correct data type.
    Unmatched candidate attributes count against correctness.

Correctness sources (in order):
1. Gold match → CORRECT
2. No gold match but supported by problem → CORRECT
3. Neither gold nor problem supports it → INCORRECT
Real-world knowledge is NOT a valid source.

Note: Only evaluate attributes within matched classes. If a class in the candidate does not match any gold
class, its attributes still count against correctness.

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


def s5_prompt(problem, gold, candidate, cot_steps):
    """
    S5 and S6 base scoring prompt.
    S6 uses this for initial scoring, then adds reflection.
    With gold reference.
    """
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


def s6_reflection_prompt(problem, gold, candidate,
                                cot_steps, previous_scores,
                                round_num):
    """
    S6 — Reflection prompt. With gold reference.
    Round 1 always runs. Round 2 only if delta > DELTA.
    """
    comp = previous_scores['attr_completeness']
    corr = previous_scores['attr_correctness']

    return f"""You previously evaluated the ATTRIBUTES in this
diagram (reflection round {round_num}) and produced:

- Attribute Completeness: {comp}
- Attribute Correctness:  {corr}

Please reflect carefully using the gold reference:

For Attribute Completeness:
1. For each matched class pair, list every attribute
   in the gold class.
2. Check which gold attributes are present in the
   corresponding candidate class.
3. Count across all matched classes:
   matched / total gold attributes
4. Verify and revise your completeness score.

For Attribute Correctness:
1. For each candidate attribute, check if it exists
   in the gold with the correct:
   - Data type (int, String, List<T>, Date, etc.)
2. Count: correct / total candidate attributes
   (unmatched candidate attributes count as wrong)
3. Verify and revise your correctness score.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Attribute Completeness:
- Attribute Correctness:"""


## **SCORE PARSER**

In [8]:
import re

def parse_attr_scores(response):
    """
    Extract attribute completeness and correctness
    from LLM response text.
    Clamps values to [0, 1].
    Returns None for any score that cannot be parsed.
    """
    scores = {
        'attr_completeness': None,
        'attr_correctness':  None
    }

    if response is None:
        return scores

    clean_response = response.replace("**", "")

    for line in clean_response.strip().split("\n"):
        line_lower = line.lower()

        match = re.search(r'[\=\:]\s*([\d\.]+)\s*%?', line)
        if not match:
            continue

        try:
            value = float(match.group(1))

            if "%" in line:
                value = value / 100

            value = max(0.0, min(1.0, value))

            if "attribute completeness" in line_lower:
                scores['attr_completeness'] = value
            elif "attribute correctness" in line_lower:
                scores['attr_correctness'] = value

        except ValueError:
            continue

    return scores

In [9]:
import re
def parse_attr_scores(response):
    """
    Extract class completeness and correctness
    from LLM response text.
    """
    scores = {
        'attr_completeness': None,
        'attr_correctness':  None
    }
    print(response)
    print("Next###############")
    if response is None:
        return scores

    def extract_value(line):
        """
        Extract a numeric score from a single line.
        Handles decimal (0.8) and fraction (8/10).
        Returns float in [0, 1] or None.
        """
        # Try fraction first: 8/10
        fraction = re.search(r'(\d+)\s*/\s*(\d+)', line)
        if fraction:
            numerator   = float(fraction.group(1))
            denominator = float(fraction.group(2))
            if denominator > 0:
                return max(0.0, min(1.0,
                           numerator / denominator))
            return None

        # Try decimal or integer: 0.8 or 1 or 0
        decimal = re.search(r':\s*([0-1](?:\.\d+)?)', line)
        if decimal:
            return max(0.0, min(1.0,
                       float(decimal.group(1))))

        return None

    for line in response.strip().split('\n'):
        line_lower = line.lower()


        if "attribute completeness" in line_lower:
                scores['attr_completeness'] = extract_value(line)
        elif "attribute correctness" in line_lower:
                scores['attr_correctness'] = extract_value(line)

    return scores

##EVALUATOR





In [10]:
def evaluate_attributes(scenario, problem, candidate, model,
                        gold=None, cot_steps_no_gold=None,
                        cot_steps_with_gold=None):
    """
    Evaluate ATTRIBUTE dimension for one candidate
    under a given scenario.

    Returns dict:
        attr_completeness   : float or None
        attr_correctness    : float or None
        reflected           : bool
        reflection_rounds   : int
    """
    reflected         = False
    reflection_rounds = 0

    # ── S1 ──────────────────────────────────────────────
    if scenario == 's1':
        prompt   = s1_prompt(problem, candidate)
        response = call_llm(prompt, model)
        scores   = parse_attr_scores(response)

    # ── S2 ──────────────────────────────────────────────
    elif scenario == 's2':
        prompt   = s2_prompt(problem, candidate, cot_steps_no_gold)
        response = call_llm(prompt, model)
        scores   = parse_attr_scores(response)

    # ── S3 — Auto-CoT + Reflection ──────────────────────
    # Uses same CoT steps as S2 (no gold)
    elif scenario == 's3':
        # Initial scoring — same as S2
        prompt = s2_prompt(problem, candidate, cot_steps_no_gold)
        response = call_llm(prompt, model)
        scores = parse_attr_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s3_reflection_prompt(problem, candidate, cot_steps_no_gold,previous_scores, round_num)
            response = call_llm(ref_prompt, model)
            scores   = parse_attr_scores(response)

    # ── S4 ──────────────────────────────────────────────
    elif scenario == 's4':
        prompt   = s4_prompt(problem, gold, candidate)
        response = call_llm(prompt, model)
        scores   = parse_attr_scores(response)

    # ── S5 ──────────────────────────────────────────────
    elif scenario == 's5':
        prompt   = s5_prompt(problem, gold, candidate,cot_steps_with_gold)
        response = call_llm(prompt, model)
        scores   = parse_attr_scores(response)

    # ── S6 — Reference + CoT + Reflection ───────────────
    elif scenario == 's6':
        # Initial scoring — same as S5
        prompt          = s5_prompt(problem, gold, candidate,cot_steps_with_gold)
        response        = call_llm(prompt, model)
        scores          = parse_attr_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s6_reflection_prompt(problem, gold, candidate, cot_steps_with_gold,previous_scores, round_num)
            response = call_llm(ref_prompt, model)
            scores   = parse_attr_scores(response)

    else:
        print(f"  Unknown scenario: {scenario}")
        scores = {
            'attr_completeness': None,
            'attr_correctness':  None
        }

    scores['reflected']         = reflected
    scores['reflection_rounds'] = reflection_rounds
    return scores

##DATA LOADING


In [11]:
import chardet

def load_gold(filepath):
    # Detect encoding first
    with open(filepath, 'rb') as f:
        detected = chardet.detect(f.read())
    encoding = detected['encoding'] or 'windows-1252'

    gold = {}
    with open(filepath, newline='', encoding=encoding) as f:
        reader = csv.DictReader(f)
        for row in reader:
            gold[row['id']] = {
                'problem':  row['problem'],
                'solution': row['solution']
            }
    return gold


def load_candidates(filepath):
    """
    Load candidates CSV.
    Columns: id, student_answer,
             score1 (class_completeness),
             score2 (class_correctness),
             score3 (attr_completeness),
             score4 (attr_correctness),
             score5 (rel_completeness),
             score6 (rel_correctness)

    id directly matches gold file id.
    """
    candidates = []
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            candidates.append({
                'id':      row['id'],
                'diagram': row['student_answer'],
                'human_scores': {
                    'class_completeness': float(row['score1']),
                    'class_correctness':  float(row['score2']),
                    'attr_completeness':  float(row['score3']),
                    'attr_correctness':   float(row['score4']),
                    'rel_completeness':   float(row['score5']),
                    'rel_correctness':    float(row['score6'])
                }
            })
    return candidates

In [12]:

def save_results(results, output_filepath):
    """Save LLM scores + human scores to CSV."""
    fieldnames = [
        'candidate_id', 'scenario', 'model',
        'reflected', 'reflection_rounds',
        'human_attr_completeness', 'human_attr_correctness',
        'llm_attr_completeness',   'llm_attr_correctness'
    ]
    with open(output_filepath, 'w', newline='',
              encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\n  Results saved to: {output_filepath}")



In [13]:

def run_attribute_evaluation(gold_file, candidate_file,
                             scenario, model, output_file):
    """
    Main loop — evaluates ATTRIBUTE dimension for all
    candidates under a given scenario and model.
    """
    print(f"\n{'='*55}")
    print(f"  CD-EVAL — Attribute Dimension")
    print(f"  Scenario        : {scenario.upper()}")
    print(f"  Model           : {model}")
    print(f"  Max reflections : {MAX_REFLECT}")
    print(f"  Delta threshold : {DELTA}")
    print(f"{'='*55}")

    # Load data
    print("\nLoading data...")
    gold       = load_gold(gold_file)
    candidates = load_candidates(candidate_file)
    print(f"  Gold scenarios : {len(gold)}")
    print(f"  Candidates     : {len(candidates)}")

    # Generate CoT steps once if needed
    # S2 and S3 share the same no-gold steps
    # S5 and S6 share the same with-gold steps
    cot_steps_no_gold   = None
    cot_steps_with_gold = None

    if scenario in ['s2', 's3']:
        cot_steps_no_gold = generate_cot_steps_no_gold(model)

    elif scenario in ['s5', 's6']:
        cot_steps_with_gold = generate_cot_steps_with_gold(model)

    # Evaluation loop
    results = []
    print(f"\nEvaluating {len(candidates)} candidates...\n")

    for candidate in tqdm(candidates):
        print("###Item Number###")
        cid          = candidate['id']
        print(cid)
        diagram      = candidate['diagram']
        human_scores = candidate['human_scores']

        # id directly matches gold
        if cid not in gold:
            print(f"\n  Warning: no gold found for "
                  f"id={cid} — skipping.")
            continue

        problem  = gold[cid]['problem']
        solution = gold[cid]['solution']

        # Run evaluation
        llm_scores = evaluate_attributes(
            scenario            = scenario,
            problem             = problem,
            candidate           = diagram,
            model               = model,
            gold                = solution,
            cot_steps_no_gold   = cot_steps_no_gold,
            cot_steps_with_gold = cot_steps_with_gold
        )

        # Collect result row
        results.append({
            'candidate_id':    cid,
            'scenario':        scenario,
            'model':           model,
            'reflected':       llm_scores['reflected'],
            'reflection_rounds': llm_scores['reflection_rounds'],
            # human scores (attribute dimension only)
            'human_attr_completeness':
                human_scores['attr_completeness'],
            'human_attr_correctness':
                human_scores['attr_correctness'],
            # llm scores
            'llm_attr_completeness':
                llm_scores['attr_completeness'],
            'llm_attr_correctness':
                llm_scores['attr_correctness']
        })

    #    time.sleep(0.5)  # avoid rate limits

    save_results(results, output_file)

    # Summary
    reflected_count = sum(1 for r in results if r['reflected'])
    print(f"\n  Total evaluated     : {len(results)}")
    print(f"  Reflection triggered: {reflected_count} "
          f"({100*reflected_count//max(len(results),1)}%)")


In [16]:
gold_file = "GoldDataset.csv"
candidate_file= "CandidatesDataset.csv"
scenario= "s4"
model="gpt-5.4"
output_file="GPTAttRevisedS4.csv"

run_attribute_evaluation(gold_file, candidate_file,scenario, model, output_file)



  CD-EVAL — Attribute Dimension
  Scenario        : S4
  Model           : gpt-5.4
  Max reflections : 2
  Delta threshold : 0.1

Loading data...
  Gold scenarios : 13
  Candidates     : 13

Evaluating 13 candidates...



  0%|          | 0/13 [00:00<?, ?it/s]

###Item Number###
1


  8%|▊         | 1/13 [00:01<00:15,  1.26s/it]

- Attribute Completeness: 0.50
- Attribute Correctness: 0.29
Next###############
###Item Number###
2


...

- Attribute Completeness: 0.89
- Attribute Correctness: 0.80
Next###############
###Item Number###
12


 92%|█████████▏| 12/13 [00:13<00:01,  1.07s/it]

- Attribute Completeness: 0.31
- Attribute Correctness: 0.31
Next###############
###Item Number###
13


100%|██████████| 13/13 [00:14<00:00,  1.10s/it]

- Attribute Completeness: 0.83
- Attribute Correctness: 0.71
Next###############

  Results saved to: /content/drive/MyDrive/Paper5LLMasEval/GPTAttRevisedS4.csv

  Total evaluated     : 13
  Reflection triggered: 0 (0%)


In [15]:
if __name__ == "__main__":

    parser = argparse.ArgumentParser(
        description="CD-EVAL: Attribute dimension evaluator"
    )
    parser.add_argument(
        "--gold", required=True,
        help="Path to gold CSV (columns: id, problem, solution)"
    )
    parser.add_argument(
        "--candidates", required=True,
        help="Path to candidates CSV"
    )
    parser.add_argument(
        "--scenario", required=True,
        choices=['s1', 's2', 's3', 's4', 's5', 's6'],
        help="Evaluation scenario"
    )
    parser.add_argument(
        "--model", default="gpt-4",
        choices=list(MODELS.keys()),
        help="LLM to use as evaluator"
    )
    parser.add_argument(
        "--output", required=True,
        help="Path to output CSV file"
    )

    args = parser.parse_args()

    run_attribute_evaluation(
        gold_file      = args.gold,
        candidate_file = args.candidates,
        scenario       = args.scenario,
        model          = args.model,
        output_file    = args.output
    )


usage: colab_kernel_launcher.py [-h] --gold GOLD --candidates CANDIDATES
                                --scenario {s1,s2,s3,s4,s5,s6}
                                [--model {gpt-5.4,gpt-5.4-mini,claude-opus-4-6,claude-sonnet-4-5}]
                                --output OUTPUT
colab_kernel_launcher.py: error: the following arguments are required: --gold, --candidates, --scenario, --output
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --gold, --candidates, --scenario, --output

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3168/2087904280.py", line 33, in <cell line: 0>
    args = parser.parse_args()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1904, in parse_args
    args, argv = sel

TypeError: object of type 'NoneType' has no len()